# RFModelQ2 — Supervised Text Classification (Random Forest)

## Setup

In [1]:
import joblib
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

PART_B_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "data" / "yelp_review_full_raw_30k.csv").exists())
DATA_FILE = PART_B_DIR / "data" / "yelp_clean.csv"   # cleaned Yelp reviews from Q1
MODEL_DIR = PART_B_DIR / "models"

## Load data

In [2]:
# --- Load the cleaned Yelp reviews and drop rows with no usable text ---
df = pd.read_csv(DATA_FILE, usecols=["clean_text", "sentiment"])
df = df.dropna(subset=["clean_text", "sentiment"]).copy()
df["clean_text"] = df["clean_text"].astype(str).str.strip()
df = df[df["clean_text"] != ""]

print(f"Reviews: {len(df)}")
print(df["sentiment"].value_counts().to_string())

Reviews: 29998
sentiment
positive    12000
negative    11998
neutral      6000


## Train/test split

In [3]:
# --- Stratified 80:20 split. The same random_state is used in Q3 and Q4 so every
#     number reported in Part B refers to the identical held-out test set. ---
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["sentiment"],
    test_size=0.2, random_state=42, stratify=df["sentiment"]
)
print(f"Train: {len(X_train)}   Test: {len(X_test)}")

Train: 23998   Test: 6000


## Baseline pipeline

In [4]:
# --- Baseline pipeline built entirely from scikit-learn defaults, left untuned on purpose ---
# tfidf : no vocabulary cap, unigrams only, raw (non-sublinear) term frequency
# clf   : 100 trees, unlimited depth, sqrt(n_features) candidates per split, no class weighting
# This is the reference point that the Q3 hyperparameter search has to beat.
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

## Train

In [5]:
# --- Fit, save for the Q3/Q4 comparison, then predict on the held-out test set ---
pipeline.fit(X_train, y_train)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, MODEL_DIR / "rf_pipeline.joblib")
preds = pipeline.predict(X_test)

print(f"TF-IDF vocabulary size: {len(pipeline.named_steps['tfidf'].vocabulary_)}")

TF-IDF vocabulary size: 38031


## Evaluate

In [6]:
# --- Per-class performance ---
print("=== Q2: Baseline Yelp Random Forest 3-Class Classification Report ===")
print(classification_report(y_test, preds, digits=4, zero_division=0))

# --- Headline measures. Macro averages matter most here because neutral has only half
#     as many reviews as negative and positive, so accuracy alone would flatter the model.
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
    y_test, preds, average="macro", zero_division=0
)
weighted_f1 = precision_recall_fscore_support(
    y_test, preds, average="weighted", zero_division=0
)[2]

print("=== Q2: Baseline Yelp Random Forest Summary ===")
print(f"Accuracy:        {accuracy_score(y_test, preds):.4f}")
print(f"Macro Precision: {macro_p:.4f}")
print(f"Macro Recall:    {macro_r:.4f}")
print(f"Macro F1-score:  {macro_f1:.4f}")
print(f"Weighted F1:     {weighted_f1:.4f}")

=== Q2: Baseline Yelp Random Forest 3-Class Classification Report ===
              precision    recall  f1-score   support

    negative     0.6867    0.8921    0.7760      2400
     neutral     0.6032    0.0317    0.0602      1200
    positive     0.7276    0.8546    0.7860      2400

    accuracy                         0.7050      6000
   macro avg     0.6725    0.5928    0.5407      6000
weighted avg     0.6863    0.7050    0.6368      6000



=== Q2: Baseline Yelp Random Forest Summary ===
Accuracy:        0.7050
Macro Precision: 0.6725
Macro Recall:    0.5928
Macro F1-score:  0.5407
Weighted F1:     0.6368
